# QC + processing notebook
This notebook is to perform QC on all the ASAP snRNA-seq 10x data we have to date

Some steps are from: https://www.sc-best-practices.org/preamble.html
### env is scvi-R 
`docker pull alexanrna/scanpy-r:1.10-v6`

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
import anndata
from scipy import stats
import glob 
import os
from matplotlib import pyplot as plt
import datetime
from scipy.sparse import csr_matrix
import warnings
import warnings
import anndata as ad
import re
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)


# 00 Data loading in

In [ ]:
names = [x for x in os.listdir('./cellranger_matrices/') if x.endswith('.matrix.mtx.gz')]

In [ ]:
names.sort()
names = [x.replace('.matrix.mtx.gz', '') for x in names]
print(names)

In [ ]:
adatas = []
for name in names:
    adata = sc.read_10x_mtx('./cellranger_matrices/', var_names = 'gene_symbols', cache=False, prefix=name + '.')
    adata.obs['sample'] = name
    adata.obs['new_index'] = adata.obs.index.astype(str)
    adata.obs['new_index'] = adata.obs['new_index'].replace('-1','-' + name, regex=True)
    adata.obs.set_index(adata.obs.new_index.astype(str),  inplace=True)
       
    adatas.append(adata)

In [ ]:
adatas

In [ ]:
len(adatas)

In [ ]:
adatas[0].obs

# 01 Data QC metrics

In [ ]:
def preprocess_adata(adata):
    """
    function to preprocess and calculate the QC metrics for each adata object

    ----------
    Parameters
    ----------
    @adata: object,required
        anndata object
    
    """
    name = adata.obs.iloc[0]['sample']
    print(name)
    sc.pp.filter_cells(adata, min_genes=0)
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes
    adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
    )
    return adata

In [ ]:
for adata in adatas:
    preprocess_adata(adata)
    

## 01.1 MAD outlier detection

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    """
    calculate median absolute deviation (MAD) to get find outliers
    """
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * stats.median_abs_deviation(M)) | (
        np.median(M) + nmads * stats.median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
for adata in adatas:
    name = adata.obs.iloc[0]['sample']
    print(name)
    adata.obs["outlier"] = (
        is_outlier(adata, "log1p_total_counts", 5)
        | is_outlier(adata, "log1p_n_genes_by_counts", 5)
        | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
        )
    print(adata.obs.outlier.value_counts())
    print("\n")

## 01.2 Mitochondria filtering

In [ ]:
for adata in adatas:
    name = adata.obs.iloc[0]['sample']
    print(name)
    
    adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 3) | (
    adata.obs["pct_counts_mt"] > 12.5
    )
    print(adata.obs.mt_outlier.value_counts())
    print("\n")

In [ ]:
adatas[0].obs

In [ ]:
for adata in adatas:
    name = adata.obs.iloc[0]['sample']
    print(name)
    sc.pl.highest_expr_genes(adata, n_top=20, )
    print("\n")

In [ ]:
df = pd.concat([x.obs for x in adatas])

In [ ]:
df = df.sort_values('sample')

In [ ]:
df

In [ ]:
value = "log1p_total_counts"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="sample", hue="sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

In [ ]:
value = "n_genes_by_counts"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="sample", hue="sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

In [ ]:
value = "pct_counts_in_top_20_genes"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="sample", hue="sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

In [ ]:
value = "n_genes_by_counts"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="sample", hue="sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

In [ ]:
value = "n_genes"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="sample", hue="sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

In [ ]:
value = "pct_counts_mt"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="sample", hue="sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

In [ ]:
for adata in adatas:
    name = adata.obs.iloc[0]['sample']
    print(name)
    sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

## 01.3 Doublet detection

Using Scrublet

In [ ]:
for adata in adatas:
    name = adata.obs.iloc[0]['sample']
    print(name)
    sc.pp.scrublet(adata)
    print("\n")

In [ ]:
adata.obs

# 02 Data combination

In [ ]:
adata = ad.concat(adatas, join='outer')

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
# number of cells in each 10x lane
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(adata.obs['sample'].value_counts())


In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

In [ ]:
adata[adata.obs.outlier]

In [ ]:
adata.obs.mt_outlier.value_counts()

In [ ]:
adata.obs.predicted_doublet.value_counts()

In [ ]:
adata[(adata.obs.predicted_doublet == True) & (adata.obs.mt_outlier == True)] # only few cells are doublets and have high mito count

In [ ]:
adata[(adata.obs.outlier == True) & (adata.obs.mt_outlier == True)] # only few cells are doublets and have high mito count

In [ ]:
adata[(adata.obs.outlier == True) & (adata.obs.predicted_doublet == True)] 

In [ ]:
adata = adata[(~adata.obs.outlier)].copy()

In [ ]:
adata

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

In [ ]:
adata = adata[adata.obs.total_counts < 75000, :]

In [ ]:
adata

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

In [ ]:
p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 10000, :]

In [ ]:
p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
adata.obs.mt_outlier.value_counts()

In [ ]:
adata = adata[adata.obs.pct_counts_mt < 12, :] # I will only filter at 12 % mito counts

In [ ]:
adata

# 03 Normalisation

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)

# 04 Feature selection

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.layers["log1p_norm"] = adata.X.copy()

# 05 PCA

In [ ]:
sc.pp.pca(adata)

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca_scatter(adata, color="total_counts")

# 06 Neighbours + UMAP

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "pct_counts_mt", "predicted_doublet", "doublet_score"],
)

In [ ]:
sc.pl.umap(
    adata,
    color=["sample"],
)

# 07 Further annotation

In [ ]:
adata.obs['experiment'] = adata.obs['sample'].str[:-1]

In [ ]:
sc.pl.umap(
    adata,
    color=["experiment"],
)

In [ ]:
adata.obs.experiment.value_counts()

In [ ]:
adata

## 07.01 Donor annotation

In [ ]:
# define function for name appending
def append_name_if_donor(row):
    if re.search(r'donor', row['donor_id'], re.IGNORECASE):
        return name + '_' + row['donor_id']
    return row['donor_id']

In [ ]:
base_dir = '/20241004_patient_demultiplexing/vireo' # change path as necessary
sub_dirs = ['snmultiomerna', 'snrna']
pattern = '*/donor_ids.tsv'

vireo_files = []

# glob files
for sub_dir in sub_dirs:
    full_pattern = os.path.join(base_dir, sub_dir, pattern)
    vireo_files.extend(glob.glob(full_pattern))

In [ ]:
len(vireo_files)

In [ ]:
df_merged = pd.DataFrame()
for file in vireo_files:
    print(file)
    name = file.replace('/20241004_patient_demultiplexing/vireo/','')
    
    # remove ['snmultiomerna', 'snrna']
    for sub_dir in sub_dirs:
        name = name.replace(sub_dir+'/', '')
    name = name.replace('/donor_ids.tsv','')
    name = name.lower()
    print('\t' + name)
    
    df = pd.read_csv(file, sep='\t')
    
    
    df['donor_id'] = df.apply(append_name_if_donor, axis=1)
    df['full_barcode'] = df['cell'].replace('-1','-'+name, regex=True)
    df['multiome'] = name
    df.index = df['full_barcode']
    #df.index = df.index.str.lower()
    
    print(f'\t{str(len(df))}')
    df_merged = pd.concat([df_merged, df])

In [ ]:
df_merged.head()

In [ ]:
df_ass = df_merged[['donor_id']]

In [ ]:
# check which multiome runs have no vireo annotation
# there are some that had too few cells, or just one donor, 
# so vireo has not been run on them

sample_vir = df_merged.multiome.unique()
sample = adata.obs['sample'].unique()
unique_to_list1 = list(set(sample_vir) - set(sample))
print(f"Elements in vireo but not in adata object: {unique_to_list1}")

unique_to_list2 = list(set(sample) - set(sample_vir))
print(f"Elements in adata object but not in vireo: {unique_to_list2}")

In [ ]:
# add selected columns to the anndata object

columns_to_add = ['prob_max', 'prob_doublet', 'n_vars', 'doublet_logLikRatio']

adata.obs = adata.obs.merge(df_merged[columns_to_add], left_index=True, right_index=True, how='left')

# add donor id as vireo_assignment
adata.obs['vireo_assignment'] = df_merged['donor_id']

In [ ]:
# add donor ASA_063, since no vireo run on this donors multiome run
multiome2annotation = {
    "mo012":"ASA_063",
}
adata.obs["vireo_assignment"] = (
    adata.obs["experiment"].map(multiome2annotation).fillna(adata.obs['vireo_assignment']) # .fillna fills the non-matching map events to the original column i.e. don't overwrite with NaN if there is no match in the cluster2annotation dict
)



In [ ]:
adata.obs['vireo_assignment'].nunique()

In [ ]:
custom_palette = sns.color_palette("colorblind", 190)
sc.pl.umap(
    adata,
    color=[ "vireo_assignment"], 
    title = ["Donors"],
    palette = custom_palette
    
)

## 07.02 Region annotation

In [ ]:
# create a dictionary to map each multiome run to its brain region of origin
cluster2annotation = {
"mo001a":"Cingulate Gyrus",   
"mo001b":"Cingulate Gyrus",   
"mo001c":"Cingulate Gyrus",  
"mo002a":"Substantia Nigra",  
"mo002b":"Cingulate Gyrus",    
"mo002c":"Substantia Nigra",  
"mo002d":"Cingulate Gyrus",   
"mo003a":"Substantia Nigra",    
"mo003b":"Substantia Nigra",    # here two regions in one run
"mo004a":"Cingulate Gyrus",    
"mo004b":"Cingulate Gyrus",  # here two regions in one run
"mo005a":"Cingulate Gyrus",
"mo005b":"Substantia Nigra",    
"mo005c":"Substantia Nigra",    
"mo006a":"Cingulate Gyrus",    
"mo007a":"Cingulate Gyrus",   
"mo007b":"Cingulate Gyrus",    
"mo009a":"Substantia Nigra",     
"mo009b":"Substantia Nigra",    
"mo009c":"Cingulate Gyrus",    
"mo009d":"Cingulate Gyrus",    
"mo010a":"Cingulate Gyrus",    
"mo010b":"Cingulate Gyrus",     
"mo008a":"Cingulate Gyrus",  
"mo011a":"Cingulate Gyrus",     
"mo011b":"Cingulate Gyrus",
'mo012a':"Cingulate Gyrus",
 'mo012b':"Cingulate Gyrus",
 'mo012c':"Cingulate Gyrus",
 'mo012d':"Cingulate Gyrus",
 'mo013a':"Cingulate Gyrus",
 'mo014a':"Substantia Nigra",
 'mo014b':"Substantia Nigra",
 'mo014c':"Substantia Nigra",
 'mo014d':"Substantia Nigra",
 'mo015a':"Cingulate Gyrus",
 'mo015b':"Cingulate Gyrus",
 'mo015c':"Cingulate Gyrus",
 'mo015d':"Cingulate Gyrus",
 'mo016a':"Substantia Nigra",
 'mo016b':"Substantia Nigra",
 'mo016c':"Substantia Nigra",
 'mo016d':"Substantia Nigra",
 'mo017a':"Cingulate Gyrus",
 'mo017b':"Cingulate Gyrus",
 'mo017c':"Cingulate Gyrus",
 'mo017d':"Cingulate Gyrus",
 'mo017e':"Cingulate Gyrus",
 'mo017f':"Cingulate Gyrus",
    'mo018a':"Substantia Nigra",
    'mo018b':"Substantia Nigra",
    'mo018c':"Cingulate Gyrus",
    'mo018d':"Cingulate Gyrus",
    'mo018e':"Substantia Nigra",
    'mo018f':"Substantia Nigra",
    'mo018g':"Cingulate Gyrus",
    'mo018h':"Cingulate Gyrus",
     'mo019a':"Substantia Nigra",
     'mo019b':"Motor Cortex",
     'mo019c':"Substantia Nigra",
     'mo019d':"Motor Cortex",
'mo020a':"Substantia Nigra",
'mo020b':"Substantia Nigra",
'mo020c':"Substantia Nigra",
'mo020d':"Substantia Nigra",
'mo021a':"Cingulate Gyrus",
'mo021b':"Substantia Nigra",
'mo021c':"Cingulate Gyrus",
'mo021d':"Substantia Nigra",
'mo021e':"Cingulate Gyrus",
'mo021f':"Substantia Nigra",
'mo021g':"Cingulate Gyrus",
'mo021h':"Substantia Nigra",

"mo022a":"Substantia Nigra",
"mo022b":"Substantia Nigra",
"mo022c":"Substantia Nigra",
"rn004a":"Substantia Nigra",
"rn004b":"Substantia Nigra",   
"rn004c":"Substantia Nigra", 
"rn005a":"Cingulate Gyrus",   
"rn005b":"Cingulate Gyrus",      
"rn005c":"Substantia Nigra",  
"rn005d":"Substantia Nigra", 
"rn005e":"mix",
"rn006a":"Substantia Nigra",

}
# add a new `.obs` column called `Region_from_metadata` by mapping clusters to annotation using pandas `map` function
adata.obs["region_pre"] = (
    adata.obs["sample"].map(cluster2annotation).astype("category")
)

In [ ]:
sc.pl.umap(
    adata,
    color=["region_pre"],
)

### correct for the mixed regions
### 10x RNA mix sample first

rn005e 26b_27b_28b_29b_30b_38b_39b_40b_43b_44b_131a_132a_133a_134a_135a_136a_137a_138a_139a_140a

In [ ]:
adata.obs[adata.obs['region_pre']=='mix']

In [ ]:
adata.obs['region'] = adata.obs['region_pre']

# Apply conditions
adata.obs.loc[(adata.obs['sample'] == 'rn005e') & (adata.obs['vireo_assignment'].isin(["ASA_026","ASA_027","ASA_028","ASA_029","ASA_030","ASA_038","ASA_039","ASA_040","ASA_043","ASA_044"])), 'region'] = 'Substantia Nigra'
adata.obs.loc[(adata.obs['sample'] == 'rn005e') & (adata.obs['vireo_assignment'].isin(["ASA_131","ASA_132","ASA_133","ASA_134","ASA_135","ASA_136","ASA_137","ASA_138","ASA_139","ASA_140"])), 'region'] = 'Cingulate Gyrus'

In [ ]:
adata.obs['region'].value_counts()

In [ ]:
# the remaining mix is only doublets or unassigned
adata.obs[adata.obs['region'] == 'mix']['vireo_assignment'].value_counts()

### 10X multiome RNA mix

mo003b and mo004a have one sample from other region each - donors 23 and mo003a is from cingulate in substantia multiome and vice versa

In [ ]:
def check_columns(row):
    if (row['vireo_assignment']=='ASA_023' and row['sample']=='mo003b'):
        return 'Cingulate Gyrus'
    else:
        return row['region']


adata.obs['region'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

In [ ]:
def check_columns(row):
    if (row['vireo_assignment']=='ASA_023' and row['sample']=='mo004a'):
        return 'Substantia Nigra'
    else:
        return row['region'] 


adata.obs['region'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

In [ ]:
adata.obs['region_pre'].value_counts()

In [ ]:
adata.obs['region'].value_counts()

In [ ]:
# visual check
sc.pl.umap(adata, color = 'region_pre')
sc.pl.umap(adata, color = 'region')

In [ ]:
# remove uncorrected regions column
adata.obs = adata.obs.drop(columns=['region_pre'])

# 08 Subset to Cingulate gyrus + motor cortex and Substantia nigra and save


In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')

In [ ]:
!mkdir -p hdf5

In [ ]:
adata.X = adata.layers["counts"].copy()
# save full object
adata.write(f'./hdf5/{date}_all_10x_data_both_regions.h5ad')

In [ ]:
adata_cortex = adata[(adata.obs["region"].isin(["Cingulate Gyrus","Motor Cortex"]))]
adata_cortex

In [ ]:
adata_sn = adata[(adata.obs["region"].isin(["Substantia Nigra"]))]
adata_sn

In [ ]:
# save 
# substantia nigra
adata_sn.write(f'./hdf5/{date}_all_10x_data_sn.h5ad')
# cingulate gyrus and motor cortex
adata_cortex.write(f'./hdf5/{date}_all_10x_data_cg.h5ad')

In [ ]:
import session_info

session_info.show()

In [ ]:
pip list